# 02 — Fine-tuning no Google Colab

Treina o `pierreguillou/gpt2-small-portuguese` no corpus de receitas usando a
GPU gratuita do Colab, e publica o modelo no Hugging Face Hub.

**Por que aqui e não na máquina local:** o treino precisa manter pesos,
gradientes e estados do otimizador na memória ao mesmo tempo (~3 GB para um
modelo de 124M). Num Mac de 8 GB com outros aplicativos abertos isso vai para
o swap, e cada passo passa de 0,1 s para dezenas de segundos. Numa T4 o treino
inteiro leva poucos minutos.

**Antes de começar:** menu `Ambiente de execução` → `Alterar o tipo de ambiente`
→ **T4 GPU**.

---

## 1. Verificar a GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Instalar dependências e trazer o código

O repositório é clonado e o `src/` entra no `sys.path` — assim os módulos
`tech_challenge_fase_04.*` ficam importáveis sem precisar instalar o pacote.

In [2]:
!pip install -q "transformers>=5.0" "datasets>=4.0" accelerate huggingface_hub

!git clone -q https://github.com/denisevitoriano/tech_challenge_fase_04.git /content/projeto

import sys
sys.path.insert(0, "/content/projeto/src")

import os
os.chdir("/content/projeto")

import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())

transformers 5.13.1 | torch 2.11.0+cu128
GPU disponível: True


## 3. Conferir o corpus

O corpus versionado no repositório é usado direto. Para reconstruí-lo do zero a
partir do Wikilivros (leva ~3 min), descomente as linhas de coleta.

In [3]:
# Reconstrução completa do corpus, se quiser partir do zero:
# !python -m tech_challenge_fase_04.dados.coleta
# !python -m tech_challenge_fase_04.dados.preparo
# !python -m tech_challenge_fase_04.dados.corpus

import json

treino = [json.loads(l) for l in open("data/processed/treino.jsonl", encoding="utf-8")]
validacao = [json.loads(l) for l in open("data/processed/validacao.jsonl", encoding="utf-8")]
print(f"{len(treino)} exemplos de treino | {len(validacao)} de validação")
print("\n--- exemplo ---\n")
print(treino[0]["texto"][:500])

4504 exemplos de treino | 508 de validação

--- exemplo ---

<|receita|>
MODO: por-titulo
CATEGORIA: Geral
TÍTULO: Aluá
INGREDIENTES:
- 1 abacaxi médio
- 2 litros de água
- Açúcar mascavado (opcional)
- Cravo-da-índia a gosto
- 1 colher de chá de gengibre ralado (opcional)
PREPARO:
1. Cortam-se fatias grossas da casca de um abacaxi (ananás)
2. Colocam-se as cascas em uma panela e cobre-se com água. Deixa-se fermentar durante pelo menos um dia (quanto mais tempo as cascas ficarem a fermentar, mais alcoólica ficará a bebida)
3. Coa-se a bebida e adicionam-s


## 4. Perplexidade do modelo base

Medida **antes** do ajuste, para servir de linha de base. Sem esse número, dizer
que "o fine-tuning funcionou" é afirmação sem evidência.

Os tokens especiais são removidos aqui: o modelo base nunca os viu, e cobrá-lo
por prever um token desconhecido mediria o vocabulário novo, não a língua.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from tech_challenge_fase_04.avaliacao.metricas import perplexidade
from tech_challenge_fase_04.dados.corpus import FIM, INICIO
from tech_challenge_fase_04.modelo import MODELO_BASE

def sem_marcadores(exemplos):
    return [e["texto"].replace(INICIO, "").replace(FIM, "").strip() for e in exemplos]

textos_validacao = sem_marcadores(validacao)

tok_base = AutoTokenizer.from_pretrained(MODELO_BASE)
mod_base = AutoModelForCausalLM.from_pretrained(MODELO_BASE).to("cuda")

ppl_base = perplexidade(mod_base, tok_base, textos_validacao)
print(f"perplexidade do modelo BASE: {ppl_base:.2f}")

del mod_base
torch.cuda.empty_cache()

[transformers] GPT2LMHeadModel LOAD REPORT from: pierreguillou/gpt2-small-portuguese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


perplexidade do modelo BASE: 37.10


## 5. Treinar

Batch 8 com 2 passos de acumulação (batch efetivo 16, o mesmo planejado para o
treino local) e `fp16` ligado, que a T4 acelera bem.

In [5]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

from tech_challenge_fase_04.modelo.treino import treinar

metricas = treinar(epocas=3, lr=5e-5, batch=8, acumulacao=2, fp16=True)
print(metricas)

[transformers] GPT2LMHeadModel LOAD REPORT from: pierreguillou/gpt2-small-portuguese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Epoch,Training Loss,Validation Loss
1,1.671989,1.582726
2,1.561868,1.500083
3,1.541544,1.489435


[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training Loss,Validation Loss,Epoch
1.541544,1.489435,3


{'eval_loss': 1.4894347190856934, 'perplexidade': 4.43458802237858}


## 6. Comparar com a linha de base

Mesmos textos, mesma função, dois modelos. A queda de perplexidade é o efeito
do ajuste.

In [6]:
from tech_challenge_fase_04.modelo import DIRETORIO_MODELO

tok_ajustado = AutoTokenizer.from_pretrained(DIRETORIO_MODELO)
mod_ajustado = AutoModelForCausalLM.from_pretrained(DIRETORIO_MODELO).to("cuda")

ppl_ajustado = perplexidade(mod_ajustado, tok_ajustado, textos_validacao)

print(f"base:     {ppl_base:8.2f}")
print(f"ajustado: {ppl_ajustado:8.2f}")
print(f"redução:  {(1 - ppl_ajustado / ppl_base):8.1%}")

import json
from pathlib import Path
Path("reports").mkdir(exist_ok=True)
Path("reports/perplexidade.json").write_text(
    json.dumps({"base": ppl_base, "ajustado": ppl_ajustado}, indent=2), encoding="utf-8"
)

base:        37.10
ajustado:     5.14
redução:     86.1%


65

## 7. Gerar algumas receitas

Teste rápido de sanidade nos três modos antes de publicar.

In [7]:
from tech_challenge_fase_04.modelo.geracao import formatar_para_leitura, livre, por_ingredientes, por_titulo

for rotulo, receitas in [
    ("POR TÍTULO", por_titulo("Bolo de banana com canela", "Bolos", seed=1)),
    ("POR INGREDIENTES", por_ingredientes(["3 ovos", "1 lata de leite condensado", "1 xícara de coco ralado"], seed=2)),
    ("SURPREENDA-ME", livre("Doces", seed=3)),
]:
    print("=" * 70)
    print(rotulo)
    print("=" * 70)
    print(formatar_para_leitura(receitas[0]))
    print()

POR TÍTULO
## Bolo de banana com canela
*Bolos*

**Ingredientes**

- 2 garrafas (tipo americano) de leite condensado
- 4 garrafas (tipo americano) de açúcar

**Modo de preparo**

1. Misture bem a canela em uma panela e acrescente o leite e deixe cozinhar por um período de uns 35 minutos, quando retire a água do fogo. Deixe descansar por 30 minutos, no liquidificador ou numa panela de pressão alta
2. Põe na geladeira até endurecer
3. Leve ao congelador para pré-aquecer

POR INGREDIENTES
## Doce de cacau
*Outros Doces*

**Ingredientes**

- 3 ovos
- 1 lata de leite condensado
- 1 xícara de coco ralado

**Modo de preparo**

1. Misture bem o leite e o coco, amasse bem e coloque-os no liquidificador
2. Levar ao fogo brando e mexa até dourar
3. Sirva com queijo parmesão picada ou frango desfiado

SURPREENDA-ME
## Chocolate branco feito em panela de chocolate
*Doces*

**Ingredientes**

- 1/2 kg de chocolate meio amargo
- 5 gemas
- 1 colher (sopa) de fermento em pó

**Modo de preparo**

1. Bata

## 8. Publicar no Hugging Face Hub

O modelo tem ~500 MB — não cabe no git, e o Streamlit Cloud precisa buscá-lo de
algum lugar. O Hub resolve os dois problemas.

Gere um token de **escrita** em
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens):
clique em *New token*, escolha o tipo **Write**, e copie o valor (ele só é
exibido uma vez).

**Onde colocar o token — duas opções:**

1. **Secrets do Colab (recomendado).** No painel esquerdo, clique no ícone de
   chave 🔑 → *Adicionar novo secret* → nome `HF_TOKEN`, valor = seu token →
   ligue *Acesso ao notebook*. O token fica guardado na sua conta do Colab,
   fora do notebook.
2. **Widget de login.** Se não houver secret, a célula abaixo abre uma caixa
   com um campo *Token*. Cole ali e clique em *Login*.

⚠️ **Nunca cole o token dentro de uma célula de código.** O notebook é salvo
com o conteúdo das células e vai para o GitHub — o token vazaria publicamente,
dando a qualquer pessoa permissão de escrita na sua conta do Hugging Face.

In [8]:
from huggingface_hub import login

# Tenta primeiro o secret do Colab; se não existir, cai no widget de login.
try:
    from google.colab import userdata

    login(token=userdata.get("HF_TOKEN"))
    print("autenticado pelo secret HF_TOKEN do Colab")
except Exception:
    from huggingface_hub import notebook_login

    notebook_login()

In [9]:
USUARIO_HF = "denisevitoriano"
NOME_MODELO = "receitas-gpt2-pt"

REPO_MODELO = f"{USUARIO_HF}/{NOME_MODELO}"

mod_ajustado.push_to_hub(REPO_MODELO)
tok_ajustado.push_to_hub(REPO_MODELO)

print(f"publicado em https://huggingface.co/{REPO_MODELO}")

publicado em https://huggingface.co/denisevitoriano/receitas-gpt2-pt


## 9. Cartão do modelo

Documenta o que é, com que dados foi treinado e quais são os limites. Boa
prática de engenharia de ML — e material pronto para o vídeo.

In [10]:
CARTAO = f'''---
language: pt
license: cc-by-sa-3.0
base_model: pierreguillou/gpt2-small-portuguese
tags:
  - text-generation
  - portuguese
  - receitas
  - culinaria
pipeline_tag: text-generation
---

# Receitas GPT-2 PT

GPT-2 small (124M) em português, ajustado para gerar receitas culinárias.

Projeto do Tech Challenge Fase 4 — Pós-graduação em Machine Learning Engineering.

## Dados

2.506 receitas extraídas do [Livro de receitas](https://pt.wikibooks.org/wiki/Livro_de_receitas)
do Wikilivros (CC-BY-SA 3.0). Split treino/validação feito **por página de
origem**, para que variações da mesma receita não vazem entre os conjuntos.

## Resultados

| Métrica | Modelo base | Ajustado |
|---|---|---|
| Perplexidade (validação) | {ppl_base:.2f} | {ppl_ajustado:.2f} |

## Formato

O modelo espera o formato de prompt abaixo, com dois modos de uso:

```
<|receita|>
MODO: por-titulo
CATEGORIA: Bolos
TÍTULO: Bolo de fubá
INGREDIENTES:
```

```
<|receita|>
MODO: por-ingredientes
INGREDIENTES:
- 3 ovos
- 1 xícara de açúcar
CATEGORIA:
```

A geração encerra no token `<|fim|>`.

## Limitações

Modelo pequeno treinado em corpus pequeno. As receitas são plausíveis na forma,
mas **não foram testadas na prática** — quantidades, tempos e temperaturas podem
estar errados. Uso recreativo e educacional.

## Código

https://github.com/denisevitoriano/tech_challenge_fase_04
'''

from huggingface_hub import ModelCard
ModelCard(CARTAO).push_to_hub(REPO_MODELO)
print("cartão publicado")

cartão publicado


## 10. Próximo passo

Com o modelo publicado, o app Streamlit já consegue carregá-lo. Defina o secret
`MODELO_HF` no Streamlit Cloud com o valor `denisevitoriano/receitas-gpt2-pt` e
faça o deploy do `app.py`.

O notebook `03_avaliacao.ipynb` faz a avaliação completa — originalidade,
diversidade e a varredura de temperatura.

In [11]:
# Baixe os arquivos de métrica para anexar ao repositório, se quiser.
from google.colab import files
files.download("reports/perplexidade.json")